In [1]:
!pip install -q mediapipe
!wget -q -O hand_landmarker.task \
    https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 100.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 15.7 MB/s eta 0:00:00


In [2]:
import mediapipe as mp
print("mediapipe version:", mp.__version__)
print("has solutions?", hasattr(mp, "solutions"))


mediapipe version: 0.10.33
has solutions? False


In [ ]:
import cv2, math, time, base64, numpy as np
from base64 import b64decode
from IPython.display import Javascript, display, HTML
from google.colab.output import eval_js
import mediapipe as mp

BaseOptions           = mp.tasks.BaseOptions
HandLandmarker        = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode     = mp.tasks.vision.RunningMode
HAND_CONNECTIONS = [
    (c.start, c.end)
    for c in mp.tasks.vision.HandLandmarksConnections.HAND_CONNECTIONS
]


# Dashboard

def setup_enhanced_dashboard(width=640, height=480):
    display(HTML(f'''
        <div id="container" style="display:flex;background:#1a1a1a;padding:20px;border-radius:15px;width:fit-content;margin:auto;box-shadow:0 10px 30px rgba(0,0,0,0.5);">
            <div style="position:relative;">
                <img id="mp_display" width="{width}" height="{height}" style="border-radius:10px;background:#000;"/>
                <div id="gesture_pill" style="position:absolute;top:20px;left:20px;padding:8px 20px;border-radius:50px;background:rgba(0,0,0,0.6);color:white;font-family:sans-serif;font-weight:bold;font-size:20px;border:2px solid #fff;">
                    No Hand
                </div>
            </div>
            <div style="width:220px;margin-left:20px;font-family:'Segoe UI',Tahoma,sans-serif;color:#ddd;">
                <h3 style="margin-top:0;color:#00d1b2;">METRICS</h3>
                <div style="margin-bottom:15px;">
                    <small style="color:#888;">PERFORMANCE</small>
                    <div id="fps_val" style="font-size:24px;font-weight:bold;">0 <span style="font-size:14px;font-weight:normal;">FPS</span></div>
                </div>
                <div style="margin-bottom:15px;">
                    <small style="color:#888;">CONFIDENCE</small>
                    <div style="width:100%;background:#333;height:10px;border-radius:5px;margin-top:5px;">
                        <div id="conf_bar" style="width:0%;height:100%;background:#00d1b2;border-radius:5px;transition:width 0.1s;"></div>
                    </div>
                    <div id="conf_val" style="text-align:right;font-size:14px;margin-top:2px;">0%</div>
                </div>
                <div style="margin-bottom:15px;">
                    <small style="color:#888;">HANDEDNESS</small>
                    <div id="hand_val" style="font-size:18px;font-weight:bold;">--</div>
                </div>
            </div>
        </div>
    '''))
    js_code = """
    (function() {
        window.startCam = async function(w, h) {
            const stream = await navigator.mediaDevices.getUserMedia({video: {width: w, height: h}});
            const video = document.createElement('video');
            video.srcObject = stream;
            await video.play();
            window._video      = video;
            window._canvas     = document.createElement('canvas');
            window._canvas.width  = w;
            window._canvas.height = h;
            window._ctx        = window._canvas.getContext('2d');
            window._displayImg = document.getElementById('mp_display');
            window._fps        = document.getElementById('fps_val');
            window._confBar    = document.getElementById('conf_bar');
            window._confVal    = document.getElementById('conf_val');
            window._handVal    = document.getElementById('hand_val');
            window._pill       = document.getElementById('gesture_pill');
        };
        window.updateDashboard = function(base64Data, fps, conf, gesture, hand) {
            window._displayImg.src = base64Data;
            window._fps.innerHTML  = `${fps} <span style="font-size:14px;font-weight:normal;">FPS</span>`;
            window._confBar.style.width = `${conf * 100}%`;
            window._confVal.innerText   = `${Math.round(conf * 100)}%`;
            window._handVal.innerText   = hand;
            window._pill.innerText      = gesture;
            if      (gesture === "OPEN")      window._pill.style.borderColor = "#48c774";
            else if (gesture === "CLOSED")    window._pill.style.borderColor = "#ff3860";
            else if (gesture === "THUMBS UP") window._pill.style.borderColor = "#ffdd57";
            else                              window._pill.style.borderColor = "#fff";
        };
        window.captureFrame = function(q) {
            q = q || 0.7;
            if (!window._video || !window._ctx) return null;
            window._ctx.drawImage(window._video, 0, 0);
            return window._canvas.toDataURL('image/jpeg', q);
        };
        window.stopCam = function() {
            if (window._video && window._video.srcObject) {
                window._video.srcObject.getTracks().forEach(track => track.stop());
                window._video.srcObject = null;
            }
        };
    })();
    """
    display(Javascript(js_code))
    eval_js(f"startCam({width}, {height})")


# Draw landmarks 

def draw_landmarks_cv2(frame, landmarks, connections, show_indices=True):
    h, w, _ = frame.shape
    pixel_landmarks = []
    for lm in landmarks:
        x, y = int(lm.x * w), int(lm.y * h)
        pixel_landmarks.append((x, y))

    for idx, point in enumerate(pixel_landmarks):
        cv2.circle(frame, point, 5, (47, 116, 237), -1)
        if show_indices:
            cv2.putText(frame, str(idx), (point[0] + 5, point[1] - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.3, (0, 0, 0), 1)

    for connection in connections:
        start_idx, end_idx = connection
        start_point = pixel_landmarks[start_idx]
        end_point = pixel_landmarks[end_idx]
        cv2.line(frame, start_point, end_point, (157, 59, 42), 2)


# Gesture classifier

def classify_gesture_with_metrics(landmarks):
    def dist(lm1, lm2):
        return math.hypot(lm1.x - lm2.x, lm1.y - lm2.y)

    wrist = landmarks[0]
    tips = [8, 12, 16, 20]
    mcps = [5, 9, 13, 17]

    ratios = []
    for tip, mcp in zip(tips, mcps):
        d_tip = dist(wrist, landmarks[tip])
        d_mcp = dist(wrist, landmarks[mcp])
        ratios.append(d_tip / d_mcp if d_mcp > 0 else 0)

    threshold = 1.3
    fingers_extended = sum(1 for r in ratios if r > threshold)

    d_thumb_tip = dist(wrist, landmarks[4])
    d_thumb_mcp = dist(wrist, landmarks[2])
    thumb_ratio = d_thumb_tip / d_thumb_mcp if d_thumb_mcp > 0 else 0
    thumb_extended = thumb_ratio > threshold

    thumb_pointing_up = (landmarks[4].y < landmarks[5].y and landmarks[4].y < landmarks[9].y
                         and landmarks[2].y < landmarks[9].y)

    state = "UNKNOWN"
    confidence = 0.0
    max_ext = 2.0
    min_ext = 1.0

    if fingers_extended == 4 and thumb_extended:
        state = "OPEN"
        avg_ratio = (sum(ratios) + thumb_ratio) / 5.0
        confidence = (avg_ratio - threshold) / (max_ext - threshold)

    elif fingers_extended == 0 and not thumb_extended:
        state = "CLOSED"
        avg_ratio = (sum(ratios) + thumb_ratio) / 5.0
        confidence = (threshold - avg_ratio) / (threshold - min_ext)

    elif fingers_extended == 0 and thumb_extended and thumb_pointing_up:
        state = "THUMBS UP"
        avg_ratio_closed = sum(ratios) / 4.0
        conf_fingers_closed = (threshold - avg_ratio_closed) / (threshold - min_ext)
        conf_thumb_open = (thumb_ratio - threshold) / (max_ext - threshold)
        confidence = (conf_fingers_closed + conf_thumb_open) / 2.0

    confidence = max(0.0, min(1.0, confidence))
    return state, confidence


# Frame reader

def read_frame():
    js_data = eval_js('captureFrame()')
    if not js_data:
        return None
    np_arr = np.frombuffer(b64decode(js_data.split(',')[1]), np.uint8)
    return cv2.imdecode(np_arr, cv2.IMREAD_COLOR)


# Main loop

setup_enhanced_dashboard()
time.sleep(2)

options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='hand_landmarker.task'),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=1,
    min_hand_detection_confidence=0.7,
    min_hand_presence_confidence=0.7,
    min_tracking_confidence=0.7
)

prev_time = time.time()
start_ms  = int(time.time() * 1000)

try:
    with HandLandmarker.create_from_options(options) as landmarker:
        for _ in range(1000):
            frame_bgr = read_frame()
            if frame_bgr is None:
                break

            curr_time    = time.time()
            fps          = int(1 / max(curr_time - prev_time, 1e-6))
            prev_time    = curr_time
            timestamp_ms = int(curr_time * 1000) - start_ms

            # Convert frame_bgr to an mp.Image and run detection.
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
            result = landmarker.detect_for_video(mp_image, timestamp_ms)

            label, conf, handedness = "No Hand", 0.0, "--"

            if result and result.hand_landmarks:
                lms         = result.hand_landmarks[0]
                label, conf = classify_gesture_with_metrics(lms)
                handedness  = result.handedness[0][0].category_name
                draw_landmarks_cv2(frame_bgr, lms, HAND_CONNECTIONS)

            _, buf   = cv2.imencode(".jpg", frame_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), 75])
            data_url = "data:image/jpeg;base64," + base64.b64encode(buf).decode("ascii")
            eval_js(f"updateDashboard('{data_url}', {fps}, {conf}, '{label}', '{handedness}')")
            time.sleep(0.01)

except KeyboardInterrupt:
    pass

eval_js("stopCam()")
print("Done.")

<IPython.core.display.Javascript object>

Done.
